# Pipeline Output - Database Inspector

This notebook displays the contents of the SQLite database produced by the pipeline.

In [123]:
import sqlite3
import pandas as pd

DB_PATH = "../data/news.db"

## News Table

In [124]:
conn = sqlite3.connect(DB_PATH)
news = pd.read_sql("SELECT * FROM news", conn)
conn.close()
news

,id,symbol,date,headline,text,source,url,sentiment_label,confidence,finbert_score
0,1,AAPL,2026-07-29,Apple Could Swing $170 Billion In Value After ...,<p>Earnings are back in the driver’s seat on T...,OpenBB,https://www.benzinga.com/markets/options/26/07...,neutral,0.000000,0.000000
1,2,AAPL,2026-07-29,"Apple Q3 Preview: Stock Hits All-Time Highs, W...",<p>Technology giant <strong>Apple Inc</strong>...,OpenBB,https://www.benzinga.com/trading-ideas/preview...,neutral,0.000000,0.000000
2,3,AAPL,2026-07-29,Apple Stock Hits All-Time Highs: What's Happen...,"<p><strong>Apple Inc </strong><a class=""ticker...",OpenBB,https://www.benzinga.com/trading-ideas/movers/...,neutral,0.000000,0.000000
3,4,AAPL,2026-07-29,10 Information Technology Stocks Whale Activit...,<p>This whale alert can help traders discover ...,OpenBB,https://www.benzinga.com/markets/options/26/07...,neutral,0.000000,0.000000
4,5,AAPL,2026-07-29,CNBC Halftime Report Final Trades: Steel Dynam...,,OpenBB,https://www.benzinga.com/quote/STLD,neutral,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
966,1000,MSFT,2026-08-07,Competitor Analysis: Evaluating Microsoft And ...,<p>In the dynamic and fiercely competitive bus...,OpenBB,https://www.benzinga.com/news/26/08/61031260/c...,neutral,0.000000,0.000000
967,1001,MSFT,2026-08-07,'Bytedance Targets Mega AI Model Nearing Anthr...,"<p><a href=""https://www.ft.com/content/9b8383b...",OpenBB,https://www.benzinga.com/news/26/08/61030521/b...,neutral,0.000000,0.000000
968,1002,MSFT,2026-07-03,EXCLUSIVE: Top 12 Most-Searched Tickers For Fi...,"<p>Each trading day, Benzinga Pro features hun...",OpenBB,https://www.benzinga.com/trading-ideas/long-id...,neutral,0.000000,0.000000
969,1003,MSFT,2026-07-03,This AI Stock Beat Nvidia by 200%—Then Zuckerb...,"<p><strong>Applied Optoelectronics, Inc.</stro...",OpenBB,https://www.benzinga.com/trading-ideas/long-id...,positive,0.804116,0.804116


## Prices Table

In [125]:
conn = sqlite3.connect(DB_PATH)
prices = pd.read_sql("SELECT * FROM prices", conn)
conn.close()
prices

,id,symbol,date,open,high,low,close,volume,daily_return,weekly_return,monthly_return
0,1,AAPL,2026-07-01,293.44,296.59,289.20,294.38,50164232.0,NaN,NaN,NaN
1,2,AAPL,2026-07-02,294.12,309.42,293.68,308.63,75400626.0,0.048407,NaN,NaN
2,3,AAPL,2026-07-06,307.36,314.20,307.00,312.66,53590000.0,0.013058,NaN,NaN
3,4,AAPL,2026-07-07,315.29,315.48,310.15,310.66,42490002.0,-0.006397,NaN,NaN
4,5,AAPL,2026-07-08,311.91,314.82,307.05,313.39,41323500.0,0.008788,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
121,122,MSFT,2026-08-24,483.21,490.61,481.86,487.31,17230800.0,0.008422,-0.019260,0.246349
122,123,MSFT,2026-08-25,485.44,492.44,484.30,491.71,19519000.0,0.009029,-0.007449,0.277401
123,124,MSFT,2026-08-26,487.85,497.40,487.31,496.37,20758900.0,0.009477,0.033351,0.254632
124,125,MSFT,2026-08-27,494.88,506.48,490.08,505.06,28688800.0,0.017507,0.048647,0.259187


## Sentiment Scores (FinBERT)

In [126]:
conn = sqlite3.connect(DB_PATH)
news_sentiment = pd.read_sql("""
    SELECT symbol, date, headline, sentiment_label, confidence, finbert_score
    FROM news
    WHERE finbert_score IS NOT NULL
""", conn)
conn.close()
news_sentiment

,symbol,date,headline,sentiment_label,confidence,finbert_score
0,AAPL,2026-07-29,Apple Could Swing $170 Billion In Value After ...,neutral,0.000000,0.000000
1,AAPL,2026-07-29,"Apple Q3 Preview: Stock Hits All-Time Highs, W...",neutral,0.000000,0.000000
2,AAPL,2026-07-29,Apple Stock Hits All-Time Highs: What's Happen...,neutral,0.000000,0.000000
3,AAPL,2026-07-29,10 Information Technology Stocks Whale Activit...,neutral,0.000000,0.000000
4,AAPL,2026-07-29,CNBC Halftime Report Final Trades: Steel Dynam...,neutral,0.000000,0.000000
...,...,...,...,...,...,...
966,MSFT,2026-08-07,Competitor Analysis: Evaluating Microsoft And ...,neutral,0.000000,0.000000
967,MSFT,2026-08-07,'Bytedance Targets Mega AI Model Nearing Anthr...,neutral,0.000000,0.000000
968,MSFT,2026-07-03,EXCLUSIVE: Top 12 Most-Searched Tickers For Fi...,neutral,0.000000,0.000000
969,MSFT,2026-07-03,This AI Stock Beat Nvidia by 200%—Then Zuckerb...,positive,0.804116,0.804116


## Correlation: Sentiment vs Returns

Ignore days where sentiment score is 0. Compute correlation between average daily sentiment and daily, weekly, and monthly returns.

In [127]:
import math
from collections import defaultdict

# Aggregate sentiment by symbol + date
daily_sentiment = (
    news_sentiment.groupby(["symbol", "date"])["finbert_score"]
    .mean()
    .reset_index()
)

sentiment_map = {}
for _, row in daily_sentiment.iterrows():
    sentiment_map[(row["symbol"], row["date"])] = row["finbert_score"]

by_symbol = defaultdict(list)
for _, row in prices.iterrows():
    by_symbol[row["symbol"]].append(row)

def corr(x, y):
    n = len(x)
    if n < 2:
        return None
    mx = sum(x) / n
    my = sum(y) / n
    num = sum((xi - mx) * (yi - my) for xi, yi in zip(x, y))
    dx = math.sqrt(sum((xi - mx) ** 2 for xi in x))
    dy = math.sqrt(sum((yi - my) ** 2 for yi in y))
    if dx == 0 or dy == 0:
        return None
    return num / (dx * dy)

print("Correlation by symbol (ignoring sentiment_score == 0):")
print(f"{'Symbol':<8} {'Daily':>10} {'Weekly':>10} {'Monthly':>10} {'N':>5}")
print("-" * 45)

for symbol, rows in sorted(by_symbol.items()):
    rows.sort(key=lambda x: x["date"])
    daily_x, daily_y = [], []
    weekly_x, weekly_y = [], []
    monthly_x, monthly_y = [], []

    for row in rows:
        score = sentiment_map.get((symbol, row["date"]))
        if score is None or score == 0:
            continue
        if row.get("daily_return") is not None:
            daily_x.append(score)
            daily_y.append(row["daily_return"])
        if row.get("weekly_return") is not None:
            weekly_x.append(score)
            weekly_y.append(row["weekly_return"])
        if row.get("monthly_return") is not None:
            monthly_x.append(score)
            monthly_y.append(row["monthly_return"])

    d = corr(daily_x, daily_y)
    w = corr(weekly_x, weekly_y)
    m = corr(monthly_x, monthly_y)

    d_str = f"{d:>10.3f}" if d is not None else "N/A"
    w_str = f"{w:>10.3f}" if w is not None else "N/A"
    m_str = f"{m:>10.3f}" if m is not None else "N/A"
    n = len(daily_x)

    print(f"{symbol:<8} {d_str:>10} {w_str:>10} {m_str:>10} {n:>5}")

Correlation by symbol (ignoring sentiment_score == 0):
Symbol        Daily     Weekly    Monthly     N
---------------------------------------------
AAPL            nan        nan        nan    41
GOOGL           N/A        N/A        N/A     0
MSFT            nan        nan        nan    42


## Correlation & Distribution by Equity

Each row corresponds to one equity. Left: bucket heatmap (15 equally spaced buckets) showing the percentage of data points per sentiment/return bin (sums to 100%), with % text in each cell. Log returns are capped at 5th/95th percentiles. Right: correlation bar chart for all return types. Use the dropdown to switch between daily, weekly, and monthly.

In [128]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

conn = sqlite3.connect(DB_PATH)
prices_df = pd.read_sql("SELECT * FROM prices", conn)
conn.close()
prices_df["date"] = pd.to_datetime(prices_df["date"])
news_sentiment["date"] = pd.to_datetime(news_sentiment["date"])

daily_sentiment = (
    news_sentiment.groupby(["symbol", "date"])["finbert_score"]
    .mean()
    .reset_index()
)

merged = prices_df.merge(daily_sentiment, on=["symbol", "date"], how="inner")
merged = merged[merged["finbert_score"] != 0].copy()

for rtype in ["daily_return", "weekly_return", "monthly_return"]:
    col = f"log_{rtype}"
    merged[col] = np.where(
        (merged[rtype].notna()) & (merged[rtype] > -1),
        np.log1p(merged[rtype]),
        np.nan,
    )

symbols = sorted(merged["symbol"].unique().tolist())
symbols = [s for s in symbols if s in news_sentiment["symbol"].values]
return_cols = ["log_daily_return", "log_weekly_return", "log_monthly_return"]
return_labels = ["Daily", "Weekly", "Monthly"]

n_buckets = 15

fig = make_subplots(
    rows=len(symbols),
    cols=2,
    subplot_titles=[title for pair in zip([f"{s} - Distribution" for s in symbols], [f"{s} - Correlation" for s in symbols]) for title in pair],
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
    specs=[[{"type": "heatmap"}, {"type": "bar"}] for _ in symbols],
)

corr_bars = []
for symbol in symbols:
    symbol_data = merged[merged["symbol"] == symbol].copy()
    corr_vals = []
    for rcol, rlabel in zip(return_cols, return_labels):
        col_data = symbol_data[[rcol, "finbert_score"]].dropna()
        if len(col_data) < 2:
            corr_vals.append(0)
            continue
        x = col_data["finbert_score"].values
        y = col_data[rcol].values
        mx, my = x.mean(), y.mean()
        num = ((x - mx) * (y - my)).sum()
        dx, dy = np.sqrt(((x - mx) ** 2).sum()), np.sqrt(((y - my) ** 2).sum())
        corr_vals.append(num / (dx * dy) if dx and dy else 0)
    corr_bars.append(corr_vals)

visibility_daily = []
visibility_weekly = []
visibility_monthly = []

for idx, symbol in enumerate(symbols, start=1):
    symbol_data = merged[merged["symbol"] == symbol].copy()

    for rcol, rlabel in zip(return_cols, return_labels):
        col_data = symbol_data[[rcol, "finbert_score"]].dropna()
        if len(col_data) < 4:
            continue

        vals = col_data[rcol].values
        lo, hi = np.percentile(vals, 5), np.percentile(vals, 95)
        if hi <= lo:
            lo, hi = vals.min(), vals.max()
            if hi <= lo:
                hi = lo + 1e-6
        col_data[rcol] = col_data[rcol].clip(lo, hi)
        score_min, score_max = col_data["finbert_score"].min(), col_data["finbert_score"].max()
        if score_max <= score_min:
            score_max = score_min + 1e-6

        score_bin_edges = np.linspace(score_min, score_max, n_buckets + 1)
        ret_bin_edges = np.linspace(lo, hi, n_buckets + 1)
        x = col_data["finbert_score"].values
        y = col_data[rcol].values
        mask = ~(np.isnan(x) | np.isnan(y))
        x, y = x[mask], y[mask]
        H, _, _ = np.histogram2d(x, y, bins=[score_bin_edges, ret_bin_edges])
        total = H.sum()
        hist2d = (H / total * 100).T if total > 0 else H.T

        score_labels = [f"{score_bin_edges[i]:.3f} to {score_bin_edges[i+1]:.3f}" for i in range(n_buckets)]
        ret_labels = [f"{ret_bin_edges[i]:.4f} to {ret_bin_edges[i+1]:.4f}" for i in range(n_buckets)]

        z_text = [[f"{val:.1f}%" for val in row] for row in hist2d]

        visible = (rlabel == "Daily")
        fig.add_trace(
            go.Heatmap(
                z=hist2d,
                x=score_labels,
                y=ret_labels,
                colorscale="Blues",
                showscale=False,
                text=z_text,
                texttemplate="%{text}",
                textfont={"size": 8},
                visible=visible,
                hovertemplate="Score bucket: %{x}<br>Return bucket: %{y}<br>Percentage: %{z:.1f}%<extra></extra>",
            ),
            row=idx,
            col=1,
        )

        if rlabel == "Daily":
            visibility_daily.append(True)
            visibility_weekly.append(False)
            visibility_monthly.append(False)
        elif rlabel == "Weekly":
            visibility_daily.append(False)
            visibility_weekly.append(True)
            visibility_monthly.append(False)
        else:
            visibility_daily.append(False)
            visibility_weekly.append(False)
            visibility_monthly.append(True)

    fig.add_trace(
        go.Bar(
            x=return_labels,
            y=corr_bars[idx - 1],
            marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"],
            text=[f"{v:.3f}" for v in corr_bars[idx - 1]],
            textposition="outside",
            name=f"{symbol} correlation",
        ),
        row=idx,
        col=2,
    )
    visibility_daily.append(True)
    visibility_weekly.append(True)
    visibility_monthly.append(True)

fig.update_layout(
    title="Sentiment vs Return: Distribution & Correlation by Equity",
    height=450 * len(symbols),
    updatemenus=[
        dict(
            buttons=[
                dict(label="Daily", method="update", args=[{"visible": visibility_daily}]),
                dict(label="Weekly", method="update", args=[{"visible": visibility_weekly}]),
                dict(label="Monthly", method="update", args=[{"visible": visibility_monthly}]),
            ],
            direction="down",
            showactive=True,
            x=0.27,
            y=1.15,
            xanchor="left",
            yanchor="top",
        )
    ],
)
fig.update_xaxes(title_text="FinBERT Score Bucket", row=len(symbols), col=1)
fig.update_yaxes(title_text="Log Return Bucket", row=len(symbols), col=1)
fig.update_xaxes(title_text="Return Type", row=len(symbols), col=2)
fig.update_yaxes(title_text="Correlation", row=len(symbols), col=2, range=[-1, 1])
fig











## Interactive Plot: Price + Sentiment by Equity

In [129]:
import plotly.graph_objects as go

conn = sqlite3.connect(DB_PATH)
prices_df = pd.read_sql("SELECT * FROM prices", conn)
conn.close()
prices_df["date"] = pd.to_datetime(prices_df["date"])
news_sentiment["date"] = pd.to_datetime(news_sentiment["date"])

# Aggregate sentiment by symbol + date, keeping headlines
daily_sentiment = (
    news_sentiment.groupby(["symbol", "date"])
    .agg(
        finbert_score=("finbert_score", "mean"),
        headline=("headline", lambda titles: ", ".join(titles.astype(str))),
    )
    .reset_index()
)

# Derive sentiment label from mean score
daily_sentiment["sentiment_label"] = daily_sentiment["finbert_score"].apply(
    lambda s: "positive" if s > 0.2 else "negative" if s < -0.2 else "neutral"
)

# Merge prices with aggregated sentiment
merged = prices_df.merge(daily_sentiment, on=["symbol", "date"], how="left")

fig = go.Figure()

colors = {"positive": "green", "negative": "red", "neutral": "gray"}
all_symbols = sorted(prices_df["symbol"].unique().tolist())
all_symbols = [s for s in all_symbols if s in news_sentiment["symbol"].values]

for symbol in all_symbols:
    stock_data = merged[merged["symbol"] == symbol].sort_values("date")

    fig.add_trace(
        go.Scatter(
            x=stock_data["date"].to_numpy(),
            y=stock_data["close"],
            mode="lines",
            name=f"{symbol}",
            line=dict(width=2),
        )
    )

    # Add sentiment markers only for this symbol
    has_sentiment = stock_data["finbert_score"].notna()
    if has_sentiment.any():
        sentiment_data = stock_data[has_sentiment]
        fig.add_trace(
            go.Scatter(
                x=sentiment_data["date"].to_numpy(),
                y=sentiment_data["close"],
                mode="markers",
                name=f"{symbol} sentiment",
                marker=dict(
                    size=(sentiment_data["finbert_score"].abs() * 30).clip(6, 25),
                    color=[colors.get(l, "gray") for l in sentiment_data["sentiment_label"]],
                    line=dict(width=1, color="black"),
                    opacity=0.9,
                ),
                text=sentiment_data["headline"],
                hovertemplate=(
                    "%{text}<br>"
                    "Date: %{x|%Y-%m-%d}<br>"
                    "Close: %{y:.2f}<br>"
                    "Sentiment: %{customdata[0]}<br>"
                    "Score: %{customdata[1]:.2f}"
                ),
                customdata=sentiment_data[["sentiment_label", "finbert_score"]].values,
            )
        )

fig.update_layout(
    title="Price + News Sentiment by Equity",
    xaxis_title="Date",
    yaxis_title="Close Price",
    hovermode="x unified",
    height=600,
)
fig



## Quick Summary

In [130]:
print(f"News rows: {len(news)}")
print(f"Prices rows: {len(prices)}")
print("News symbols:", news["symbol"].unique().tolist() if len(news) else [])
print("Price symbols:", prices["symbol"].unique().tolist() if len(prices) else [])
print(f"FinBERT scores: {news_sentiment['finbert_score'].notna().sum()}")

News rows: 971
Prices rows: 126
News symbols: ['AAPL', 'MSFT']
Price symbols: ['AAPL', 'GOOGL', 'MSFT']
FinBERT scores: 971
